src

src/library

In [1]:
import pyomo.environ as pyomo

src/data_store/coeff

In [2]:
STAGE = 1

C1_data_x1_stage_1 = 1
C1_data_x2_stage_1 = 1
C2_data_x1_stage_1 = 3
C2_data_x2_stage_1 = 2
C1_rhs_stage_1 = 5
C2_rhs_stage_1 = 12

X1_stage_1_objective_coefficient = 6
X2_stage_1_objective_coefficient = 5

src/data_store/var_bounds

In [3]:
x1_stage_1_lower_bound = 0
x1_stage_1_upper_bound = None
x2_stage_1_lower_bound = 0
x2_stage_1_upper_bound = None

src/polytope_building

In [4]:
if STAGE == 1:
    stage_1_model = pyomo.ConcreteModel()

src/variable_store

In [5]:
stage_1_model.x1 = pyomo.Var(domain=pyomo.Reals, bounds=(x1_stage_1_lower_bound, x1_stage_1_upper_bound))
stage_1_model.x2 = pyomo.Var(domain=pyomo.Reals, bounds=(x2_stage_1_lower_bound, x2_stage_1_upper_bound))

src/constraint_store

In [6]:
def add_c1__stage_1_constraint_rule(m):
    return C1_data_x1_stage_1*m.x1 + C1_data_x2_stage_1*m.x2 <= C1_rhs_stage_1

def add_c2_stage_1_constraint_rule(m):
    return C2_data_x1_stage_1*m.x1 + C2_data_x2_stage_1*m.x2 <= C2_rhs_stage_1

src/objective store

In [7]:
stage_1_model.stage_1_x1_expr = pyomo.Expression(
    expr=X1_stage_1_objective_coefficient * stage_1_model.x1,
    doc="Objective contribution for x1 in Stage 1"
)

stage_1_model.stage_1_x2_expr = pyomo.Expression(
    expr=X2_stage_1_objective_coefficient * stage_1_model.x2,
    doc="Objective contribution for x2 in Stage 1"
)


src/orchestration

In [8]:
if STAGE == 1:
    stage_1_model.objective_registry = []

    # if stage_1_model has an attribute which containts the term stage_1 then append that attribute to the objective registry
    for attr_name in dir(stage_1_model):
        if "stage_1" in attr_name:
            attr = getattr(stage_1_model, attr_name)
            if isinstance(attr, pyomo.Expression):
                stage_1_model.objective_registry.append(attr)

    stage_1_model.objective_function = pyomo.Objective(
        expr=sum(stage_1_model.objective_registry), 
        sense=pyomo.maximize
    )

src/solution_extraction

In [9]:
if STAGE == 1:
    stage_1_model.dual = pyomo.Suffix(direction=pyomo.Suffix.IMPORT)
    stage_1_model.rc = pyomo.Suffix(direction=pyomo.Suffix.IMPORT)

src/solution

In [10]:
import sys
sys.path.append('.') # Ensure utils can be imported
from utils.create_solver import pyomo_solver_creator

# Create the solver (using your custom function)
solver = pyomo_solver_creator(solver_name='appsi_highs')

# Solve the model
results = solver.solve(stage_1_model, tee=True)

# Print the primal variables
print(f"x1 = {pyomo.value(stage_1_model.x1)}")
print(f"x2 = {pyomo.value(stage_1_model.x2)}")

# Print the objective value
print(f"Objective = {pyomo.value(stage_1_model.objective_function)}")

# Print the dual variables (pi) for the constraints
print(f"Dual for c1 (pi 1) = {stage_1_model.dual[stage_1_model.c1]}")
print(f"Dual for c2 (pi 2) = {stage_1_model.dual[stage_1_model.c2]}")

Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
LP has 0 rows; 2 cols; 0 nonzeros
Solving unconstrained LP
Solving an unconstrained LP with 2 columns

Model status        : Unbounded
Objective value     :  0.0000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


RuntimeError: A feasible solution was not found, so no solution can be loaded. If using the appsi.solvers.Highs interface, you can set opt.config.load_solution=False. If using the environ.SolverFactory interface, you can set opt.solve(model, load_solutions = False). Then you can check results.termination_condition and results.best_feasible_objective before loading a solution.